# CSE 151B — SFT evaluation harness

Evaluates **LoRA checkpoints** on frozen **`data/eval/holdout.jsonl`** (225 rows, seed 42). Same vLLM + `Judger` path as `notebooks/dev.ipynb` — **A100 bf16**, **`multi_blank`** prompts, 16k generation — with SFT monitoring metrics. Section 9 also reports eval watch sets: **Q4 long** (`data/eval/watch_q4_long.jsonl`, 30 ids) and **multi-blank ≥3** (`data/eval/watch_multi_blank_ge3.jsonl`, 20 ids) from `data/eval/watch_manifest.json`.

1. (**Colab A100**) Copy `data/eval/` holdout + watch files from Drive.
2. Load base model + optional LoRA adapter (bf16 vLLM).
3. Generate → score → extended summary.
4. Write `results/sft_eval/<run_name>/eval_<step>.json` on Drive.

Set `LORA_PATH = ""` for baseline-only eval.


## 1. Environment

**Colab A100:** run the `%pip` cell below, restart runtime, continue. **Local:** use your venv with `vllm`, `transformers`, etc. — **no `bitsandbytes`** (bf16 native load; see [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md)).


In [ ]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

## 2. Imports & configuration

SFT keys: `LORA_PATH`, `SFT_RUN_NAME`, `EVAL_STEP`, `PROMPT_VARIANT`, `MAX_TOKENS`, `BASELINE_MEAN_RESPONSE_LEN`. Uses frozen `data/eval/holdout.jsonl` — never resample here. Defaults match `notebooks/dev.ipynb` (`multi_blank`, 16k).


In [ ]:
import json
import os
import random
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

EVAL_HOLDOUT_PATH = str(REPO_ROOT / "data/eval/holdout.jsonl")
DATA_PATH         = EVAL_HOLDOUT_PATH

LORA_PATH     = "checkpoints/openr1_1k/final_adapter"
SFT_RUN_NAME  = "openr1_1k"
EVAL_STEP     = 0
BASELINE_MEAN_RESPONSE_LEN = 9000

PROMPT_VARIANT = "multi_blank"  # match dev.ipynb default (roadmap §1.3)
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
OUTPUT_PATH = str(REPO_ROOT / "results/sft_eval_results.jsonl")
MAX_TOKENS  = 16384
SEED        = 42

print(f"REPO_ROOT={REPO_ROOT}")
print(f"LORA_PATH={LORA_PATH or '(none — base model)'}")
print(f"PROMPT_VARIANT={PROMPT_VARIANT!r}  MAX_TOKENS={MAX_TOKENS}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import re
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm


## 3. Colab: copy eval holdout + watch sets from Drive

Canonical snapshots under `CSE151B/data/eval/` on Drive.


In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/eval/`.")
    DRIVE_PROJECT_ROOT = None
else:
    drive.mount("/content/drive")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CSE151B")
    DRIVE_EVAL = DRIVE_PROJECT_ROOT / "data/eval"
    DRIVE_HOLDOUT = DRIVE_EVAL / "holdout.jsonl"
    if not DRIVE_HOLDOUT.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_HOLDOUT}\n"
            "Upload frozen eval holdout to Drive (data/eval/holdout.jsonl)."
        )
    eval_dir = REPO_ROOT / "data/eval"
    eval_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_HOLDOUT, eval_dir / "holdout.jsonl")
    print(f"Copied holdout.jsonl to {(eval_dir / 'holdout.jsonl').resolve()}")

    for name in ("watch_manifest.json", "watch_q4_long.jsonl", "watch_multi_blank_ge3.jsonl"):
        src = DRIVE_EVAL / name
        if src.is_file():
            shutil.copy2(src, eval_dir / name)
            print(f"Copied {name}")

    if LORA_PATH and not Path(LORA_PATH).is_absolute():
        _lora = DRIVE_PROJECT_ROOT / LORA_PATH
        if _lora.is_dir():
            LORA_PATH = str(_lora)
            print(f"Resolved LORA_PATH={LORA_PATH}")


## 4. Load dataset

Frozen `data/eval/holdout.jsonl`. Rebuild via `scripts/build_eval_holdout.py` (not in this notebook).


In [ ]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next((d for d in data if d.get("options")), None)
free_sample = next((d for d in data if not d.get("options")), None)
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

if mcq_sample:
    print("\n── MCQ sample ──")
    print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
if free_sample:
    print("── Free-form sample ──")
    print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample and multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

## 5. Prompt construction

Same as `notebooks/dev.ipynb` §6 — controlled by `PROMPT_VARIANT` (default `"multi_blank"`): separate `\boxed{}` per `[ANS]` blank, comma-separated in blank order.


In [ ]:
_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

_MATH_MULTI_BLANK = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
    "\\boxed{}, separated by commas, in the order the blanks appear "
    "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes. "
    "For single-answer problems, use one \\boxed{}."
)

_MCQ_MULTI_BLANK = _MCQ_BASELINE

_PROMPTS = {
    "baseline": (_MATH_BASELINE, _MCQ_BASELINE),
    "multi_blank": (_MATH_MULTI_BLANK, _MCQ_MULTI_BLANK),
}
assert PROMPT_VARIANT in _PROMPTS, f"Unknown PROMPT_VARIANT {PROMPT_VARIANT!r}; choose from {list(_PROMPTS)}"
SYSTEM_PROMPT_MATH, SYSTEM_PROMPT_MCQ = _PROMPTS[PROMPT_VARIANT]


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    n_blanks = n_ans_placeholders(question)
    user = question
    if PROMPT_VARIANT == "multi_blank" and n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        user = (
            f"{question}\n\n"
            f"The problem has {n_blanks} [ANS] blanks. "
            f"After your reasoning, give {n_blanks} comma-separated \\boxed{{}} values "
            f"in order: {example}"
        )
    return SYSTEM_PROMPT_MATH, user


print(f"Active variant: {PROMPT_VARIANT!r}")

_demo_items = []
if mcq_sample:
    _demo_items.append(("MCQ", mcq_sample))
if free_sample:
    _demo_items.append(("Free-form", free_sample))
if multi_sample and multi_sample not in (mcq_sample, free_sample):
    _demo_items.append((f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample))
for label, item in _demo_items:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 300 chars) ──")
    print(usr_p[:300], "...\n")

## 6. Load model with vLLM (A100 profile)

Matches `notebooks/dev.ipynb` §7 — bf16, `max_model_len=17500`, prefix caching, chunked prefill. LoRA when `LORA_PATH` is set. See [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md).


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

USE_LORA = bool(LORA_PATH) and Path(LORA_PATH).is_dir()
if LORA_PATH and not USE_LORA:
    raise FileNotFoundError(f"LORA_PATH set but not found: {LORA_PATH}")

_llm_kwargs = dict(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.92,
    max_model_len=17500,
    trust_remote_code=True,
    max_num_seqs=128,
    max_num_batched_tokens=32768,
    enable_chunked_prefill=True,
)
if USE_LORA:
    _llm_kwargs.update(enable_lora=True, max_loras=1, max_lora_rank=64)

with _jupyter_stdout_for_vllm():
    llm = LLM(**_llm_kwargs)

lora_request = None
if USE_LORA:
    from vllm.lora.request import LoRARequest
    lora_request = LoRARequest("sft_adapter", 1, LORA_PATH)
    print(f"LoRA adapter: {LORA_PATH}")
else:
    print("No LoRA — evaluating base model.")

default_sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed=SEED,
)

print("Model loaded.")


## 7. Generate responses

Same loop as `notebooks/dev.ipynb` §8 (K=1). Checkpoint: `results/sft_eval/<run_name>/eval_<step>.responses.jsonl` — delete to regenerate.


In [ ]:
CHUNK_SIZE = len(data)

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(OUTPUT_PATH).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / "sft_eval" / SFT_RUN_NAME / f"eval_{EVAL_STEP}.responses.jsonl"
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    chunk_params = [default_sampling_params] * len(prompts)

    with _jupyter_stdout_for_vllm():
        _gen_kwargs = dict(prompts=prompts, sampling_params=chunk_params)
        if lora_request is not None:
            _gen_kwargs["lora_request"] = lora_request
        outputs = llm.generate(**_gen_kwargs)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            response = out.outputs[0].text.strip()
            completed[item["id"]] = response
            f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")


## 8. Score responses (same as starter)


In [ ]:
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
try:
    _drive_root = str(DRIVE_PROJECT_ROOT)
except NameError:
    _drive_root = ""
JUDGER_ROOT = _judger_override or _drive_root or str(REPO_ROOT)

print(f"JUDGER_ROOT={JUDGER_ROOT}")
sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 9. Extended summary


In [ ]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


def mcq_has_boxed_letter(response: str, n_options: int) -> bool:
    last = chr(64 + n_options)
    return bool(re.search(rf"\\boxed\{{[A-{last}]\}}", response, re.IGNORECASE))


mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]
multi_free = [r for r in free_res if n_ans_placeholders(next(d["question"] for d in data if d["id"] == r["id"])) > 1]
single_free = [r for r in free_res if r not in multi_free]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

mean_len = sum(len(r["response"]) for r in results) / len(results)
mcq_emission = 0.0
if mcq_res:
    emitted = 0
    for r in mcq_res:
        item = next(d for d in data if d["id"] == r["id"])
        n_opts = len(item.get("options") or [])
        if mcq_has_boxed_letter(r["response"], n_opts):
            emitted += 1
    mcq_emission = emitted / len(mcq_res) * 100

len_delta_pct = (mean_len - BASELINE_MEAN_RESPONSE_LEN) / BASELINE_MEAN_RESPONSE_LEN * 100

print("=" * 50)
print(f"SFT EVAL — {SFT_RUN_NAME} @ step {EVAL_STEP}")
print("=" * 50)
print(f"  LoRA       : {LORA_PATH or '(base model)'}")
print(f"  PROMPT     : {PROMPT_VARIANT}")
print(f"  MAX_TOKENS : {MAX_TOKENS}")
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Multi-blank: {sum(r['correct'] for r in multi_free):4d} / {len(multi_free):4d}  ({acc(multi_free):.2f}%)")
print(f"  Single-blank: {sum(r['correct'] for r in single_free):4d} / {len(single_free):4d}  ({acc(single_free):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("-" * 50)

import sys
sys.path.insert(0, str(REPO_ROOT))
from scripts.eval_watch import watch_accuracy

q4_pct, q4_ok, q4_n = watch_accuracy(results, "q4_long")
mb_pct, mb_ok, mb_n = watch_accuracy(results, "multi_blank_ge3")
print(f"  Q4 long (≥435 chars) : {q4_ok:4d} / {q4_n:4d}  ({q4_pct:.2f}%)")
print(f"  Multi-blank ≥3       : {mb_ok:4d} / {mb_n:4d}  ({mb_pct:.2f}%)")
print("-" * 50)
print(f"  MCQ \\boxed{{}} emission : {mcq_emission:.2f}%")
print(f"  Mean response length  : {mean_len:.0f} chars ({len_delta_pct:+.1f}% vs baseline {BASELINE_MEAN_RESPONSE_LEN})")
if len_delta_pct < -20:
    print("  WARNING: mean length dropped >20% — possible trace-style collapse")
print("=" * 50)

eval_record = {
    "run_name": SFT_RUN_NAME,
    "eval_step": EVAL_STEP,
    "lora_path": LORA_PATH or None,
    "prompt_variant": PROMPT_VARIANT,
    "max_tokens": MAX_TOKENS,
    "n": len(results),
    "mcq_acc": acc(mcq_res),
    "free_acc": acc(free_res),
    "multi_blank_acc": acc(multi_free),
    "single_blank_acc": acc(single_free),
    "overall_acc": acc(results),
    "q4_long_acc": q4_pct,
    "multi_blank_ge3_acc": mb_pct,
    "mcq_boxed_emission_pct": mcq_emission,
    "mean_response_len": mean_len,
    "mean_len_delta_pct": len_delta_pct,
}


## 10. Save results

Same schema as the starter (`id`, `is_mcq`, `gold`, `response`, `correct`). Uses Drive when `DRIVE_PROJECT_ROOT` exists.


In [ ]:
SAVE_EVAL = True

try:
    _eval_dir = DRIVE_PROJECT_ROOT / "results" / "sft_eval" / SFT_RUN_NAME
except NameError:
    _eval_dir = Path(REPO_ROOT) / "results" / "sft_eval" / SFT_RUN_NAME

_eval_dir.mkdir(parents=True, exist_ok=True)
eval_json_path = _eval_dir / f"eval_{EVAL_STEP}.json"
out_path = _eval_dir / f"eval_{EVAL_STEP}.jsonl"

with open(out_path, "w") as f:
    for r in results:
        record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                  "response": r["response"], "correct": r["correct"]}
        f.write(json.dumps(record) + "\n")

with open(eval_json_path, "w") as f:
    json.dump(eval_record, f, indent=2)

print(f"Saved {len(results)} records to {out_path.resolve()}")
print(f"Saved metrics to {eval_json_path.resolve()}")


In [ ]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")